# Monthly Core HR metrics

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

### Globals

In [118]:
current_month = pd.to_datetime('2024-11-01')
current_month_formatted = current_month.strftime("%b '%y")
previous_month = current_month - pd.DateOffset(months=1)
previous_month_formatted = previous_month.strftime("%b '%y")
start_of_year = current_month.replace(month=1, day=1)
end_of_year = current_month.replace(month=12, day=31)

attrition_year_label = f"{current_month.strftime('%Y')} + KFA"
attrition_year_label_2 = f"{current_month.strftime('%Y')} YTD"

In [3]:
current_month.strftime('%Y')

'2024'

### Monthly employee data

In [4]:
# declaring fields in use
fields_monthly = [
    'ds_start', 'ds', 'is_last_day_of_year', 'employee_id', 'hire_date', 'prehire_status', 'is_active',
    'termination_date', 'termination_date_coalesced', 'termination_month', 'termination_reason', 'is_termination_voluntary', 'is_terminated',
    'org_l00', 'org_l01', 'org_l02', 'org_l03', 
    'gender', 'gender_remapped', 'ethnicity', 'ethnicity_remapped', 
    'is_manager_track', 'job_track', 'job_level_idx',
    'job_level_category', 'job_level_category_ordered_w_indicators'
]

In [5]:
df_employees_plus = pd.read_csv(
    filepath_or_buffer='../transforms/exclude/employee_data_plus.csv',
    dtype='str'
)
# type casting and renaming fields
df_employees_plus['ds'] = pd.to_datetime(df_employees_plus['full_date']).dt.normalize()
df_employees_plus['ds_start'] = df_employees_plus.ds.dt.to_period('M').dt.start_time
df_employees_plus['is_last_day_of_year'] = df_employees_plus.is_last_day_of_year.astype('bool')
df_employees_plus['hire_date'] = pd.to_datetime(df_employees_plus['hire_date']).dt.normalize()
df_employees_plus['termination_date'] = pd.to_datetime(df_employees_plus['termination_date_coalesced']).dt.normalize()
df_employees_plus['termination_month'] = df_employees_plus.termination_date.dt.to_period('M').dt.start_time.dt.normalize()
df_employees_plus['is_termination_voluntary'] = df_employees_plus.is_termination_voluntary.astype('bool')
df_employees_plus['is_manager_track'] = df_employees_plus.is_manager_track == 'True'
df_employees_plus['org_l00'] = df_employees_plus['org_l00'].fillna('')
df_employees_plus['org_l01'] = df_employees_plus['org_l01'].fillna('')
df_employees_plus['org_l02'] = df_employees_plus['org_l02'].fillna('')

# deriving is_active
condition_active = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds <= df_employees_plus.termination_date)
df_employees_plus['is_active'] = condition_active

# deriving is_terminated
condition_terminated_in_current_month = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds_start <= df_employees_plus.termination_date) &\
    (df_employees_plus.ds >= df_employees_plus.termination_date)
df_employees_plus['is_terminated'] = condition_terminated_in_current_month

# reorganizing fields
df_employees_plus = df_employees_plus[fields_monthly]

df_employees_plus.dtypes

ds_start                                   datetime64[ns]
ds                                         datetime64[ns]
is_last_day_of_year                                  bool
employee_id                                        object
hire_date                                  datetime64[ns]
prehire_status                                     object
is_active                                            bool
termination_date                           datetime64[ns]
termination_date_coalesced                         object
termination_month                          datetime64[ns]
termination_reason                                 object
is_termination_voluntary                             bool
is_terminated                                        bool
org_l00                                            object
org_l01                                            object
org_l02                                            object
org_l03                                            object
gender        

In [6]:
df_employees_plus.ds.dt.to_period('Y').dt.end_time.dt.date

0         2021-12-31
1         2021-12-31
2         2021-12-31
3         2021-12-31
4         2021-12-31
             ...    
412195    2026-12-31
412196    2026-12-31
412197    2026-12-31
412198    2026-12-31
412199    2026-12-31
Name: ds, Length: 412200, dtype: object

In [7]:
condition_new_hire = (df_employees_plus.hire_date >= df_employees_plus.ds_start) &\
    (df_employees_plus.hire_date <= df_employees_plus.ds)
df_employees_plus['is_new_hire'] = condition_new_hire

In [8]:
df_employees_plus.head()

,ds_start,ds,is_last_day_of_year,employee_id,hire_date,prehire_status,is_active,termination_date,termination_date_coalesced,termination_month,...,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators,is_new_hire
0,2021-01-01,2021-01-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
1,2021-02-01,2021-02-28,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
2,2021-03-01,2021-03-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
3,2021-04-01,2021-04-30,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
4,2021-05-01,2021-05-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False


### Annual employee data

In [9]:
# declaring fields in use
fields_annually = [
    'ds_start', 'ds', 'is_last_day_of_year', 'employee_id', 'hire_date', 'prehire_status', 'is_active', 
    'termination_date', 'termination_date_coalesced', 'termination_year', 'termination_reason', 'is_termination_voluntary', 'is_terminated',
    'org_l00', 'org_l01', 'org_l02', 'org_l03', 
    'gender', 'gender_remapped', 'ethnicity', 'ethnicity_remapped', 
    'is_manager_track', 'job_track', 'job_level_idx',
    'job_level_category', 'job_level_category_ordered_w_indicators'
]

In [50]:
# extract and derive initial fields ------------------------------------------------
df_employees_plus_annual = pd.read_csv(
    filepath_or_buffer='../transforms/exclude/employee_data_plus_annual.csv',
    dtype='str'
)
# type casting and renaming fields
df_employees_plus_annual['ds'] = pd.to_datetime(df_employees_plus_annual['full_date']).dt.normalize()
df_employees_plus_annual['ds_start'] = df_employees_plus_annual.ds.dt.to_period('Y').dt.start_time
df_employees_plus_annual['is_last_day_of_year'] = df_employees_plus_annual.is_last_day_of_year.astype('bool')
df_employees_plus_annual['hire_date'] = pd.to_datetime(df_employees_plus_annual['hire_date']).dt.normalize()
df_employees_plus_annual['termination_date'] = pd.to_datetime(df_employees_plus_annual['termination_date_coalesced']).dt.normalize()
df_employees_plus_annual['termination_year'] = df_employees_plus_annual.termination_date.dt.to_period('Y').dt.start_time
df_employees_plus_annual['is_termination_voluntary'] = df_employees_plus_annual.is_termination_voluntary.astype('bool')
df_employees_plus_annual['is_manager_track'] = df_employees_plus_annual.is_manager_track.astype('bool')
df_employees_plus_annual['org_l00'] = df_employees_plus_annual['org_l00'].fillna('')
df_employees_plus_annual['org_l01'] = df_employees_plus_annual['org_l01'].fillna('')
df_employees_plus_annual['org_l02'] = df_employees_plus_annual['org_l02'].fillna('')

# deriving is_active
condition_active = (df_employees_plus_annual.ds >= df_employees_plus_annual.hire_date) &\
    (df_employees_plus_annual.ds <= df_employees_plus_annual.termination_date)
df_employees_plus_annual['is_active'] = condition_active

# deriving is_terminated
condition_terminated_in_current_year = (df_employees_plus_annual.ds >= df_employees_plus_annual.hire_date) &\
    (df_employees_plus_annual.ds_start <= df_employees_plus_annual.termination_date) &\
    (df_employees_plus_annual.ds >= df_employees_plus_annual.termination_date)
df_employees_plus_annual['is_terminated'] = condition_terminated_in_current_year

# reorganizing fields
df_employees_plus_annual = df_employees_plus_annual[fields_annually]

# is_new_hire ----------------------------------------------------------------------
condition_new_hire = (df_employees_plus_annual.hire_date >= df_employees_plus_annual.ds_start) &\
    (df_employees_plus_annual.hire_date <= df_employees_plus_annual.ds)
df_employees_plus_annual['is_new_hire'] = condition_new_hire

print(df_employees_plus_annual.dtypes)


ds_start                                   datetime64[ns]
ds                                         datetime64[ns]
is_last_day_of_year                                  bool
employee_id                                        object
hire_date                                  datetime64[ns]
prehire_status                                     object
is_active                                            bool
termination_date                           datetime64[ns]
termination_date_coalesced                         object
termination_year                           datetime64[ns]
termination_reason                                 object
is_termination_voluntary                             bool
is_terminated                                        bool
org_l00                                            object
org_l01                                            object
org_l02                                            object
org_l03                                            object
gender        

In [11]:
df_employees_plus_annual

,ds_start,ds,is_last_day_of_year,employee_id,hire_date,prehire_status,is_active,termination_date,termination_date_coalesced,termination_year,...,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators,is_new_hire
0,2021-01-01,2021-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
1,2022-01-01,2022-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
2,2023-01-01,2023-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
3,2024-01-01,2024-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
4,2025-01-01,2025-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34345,2022-01-01,2022-12-31,True,e005725,2024-11-25,Prehire,False,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3),False
34346,2023-01-01,2023-12-31,True,e005725,2024-11-25,Prehire,False,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3),False
34347,2024-01-01,2024-12-31,True,e005725,2024-11-25,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3),True
34348,2025-01-01,2025-12-31,True,e005725,2024-11-25,Not prehire,True,2260-01-01,2260-01-01,2260-01-01,...,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3),False


In [12]:
df_employees_plus_annual[(df_employees_plus_annual.employee_id == 'e000029')
                        #  & (df_employees_plus_annual.ds == '2024-12-31')
                         & (True)] \
    [['ds_start', 'ds', 'employee_id', 'hire_date', 'termination_date']]

,ds_start,ds,employee_id,hire_date,termination_date
168,2021-01-01,2021-12-31,e000029,2014-06-23,2024-03-06
169,2022-01-01,2022-12-31,e000029,2014-06-23,2024-03-06
170,2023-01-01,2023-12-31,e000029,2014-06-23,2024-03-06
171,2024-01-01,2024-12-31,e000029,2014-06-23,2024-03-06
172,2025-01-01,2025-12-31,e000029,2014-06-23,2024-03-06
173,2026-01-01,2026-12-31,e000029,2014-06-23,2024-03-06


## Monthly results

### Creating monthly dataframes

#### Monthly (overall)

In [13]:
df_monthly = pd.DataFrame({'ds': df_employees_plus.ds.unique(), 'ds_start': df_employees_plus.ds_start.unique()})


##### Deriving `n_active_employees`

In [14]:
# df_monthly['n_active_employees_by_status']
active_employees_by_status = df_employees_plus[df_employees_plus['is_active']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_monthly = df_monthly.merge(right=active_employees_by_status, how='left', on='ds')

##### Deriving `n_terminated_employees`

In [15]:
terminated_by_dates_monthly_by_status = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

df_monthly = df_monthly.merge(right=terminated_by_dates_monthly_by_status, how='left', on='ds')
df_monthly['n_terminated_employees'] = pd.to_numeric(df_monthly.n_terminated_employees, errors='coerce').fillna(0).astype('int')


##### Deriving fields for `attrition_rate`

- `avg_active_employees` - active employee count 2 month rolling (between current month and previous month)
- `attrition_rate = terminated_employees / avg_active_employees`

In [16]:
df_monthly['avg_active_employees'] = df_monthly['n_active_employees'].rolling(window=2).mean()
df_monthly['avg_active_employees'] = np.where(
    df_monthly['avg_active_employees'].isnull(),
    df_monthly['n_active_employees'],
    df_monthly['avg_active_employees']
)
df_monthly['attrition_rate'] = df_monthly['n_terminated_employees'] / df_monthly['avg_active_employees']

##### Deriving `n_new_hire_employees`

In [17]:
new_hire_employees_by_status = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')
df_monthly = df_monthly.merge(right=new_hire_employees_by_status, how='left', on='ds', )
df_monthly['n_new_hire_employees'] = df_monthly['n_new_hire_employees'].fillna(0).astype('int')

#### Monthly results by organization

##### Monthly by org_l01

Deriving counts for active, terminated, and new hire employees

In [18]:
temp_indices = ['ds', 'ds_start', 'org_l00', 'org_l01']

# deriving active employee count
df_monthly_by_org1 = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# adding placeholder field and reordering fields
df_monthly_by_org1['org_l02'] = df_monthly_by_org1['org_l01'] + ' Total'
df_monthly_by_org1 = df_monthly_by_org1[temp_indices + ['org_l02'] + ['n_active_employees']]

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees') \

# merging and typecasting
df_monthly_by_org1 = pd.merge(left=df_monthly_by_org1, right=new_hires_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org1['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_org1.n_new_hire_employees, errors='coerce')
df_monthly_by_org1['n_new_hire_employees'] = df_monthly_by_org1.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_org1 = pd.merge(left=df_monthly_by_org1, right=terminated_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org1['n_terminated_employees'] = pd.to_numeric(df_monthly_by_org1['n_terminated_employees'], errors='coerce')
df_monthly_by_org1['n_terminated_employees'] = df_monthly_by_org1['n_terminated_employees'].fillna(0).astype('int')


Deriving two-month average of active employees

In [19]:
df_temp_reindexed = df_monthly_by_org1 \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0,1], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_org1 = pd.merge(left=df_monthly_by_org1, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_org1['avg_active_employees'] = np.where(
    df_monthly_by_org1['avg_active_employees'].isnull(),
    df_monthly_by_org1['n_active_employees'],
    df_monthly_by_org1['avg_active_employees']
)

Deriving `attrition_rate`

In [20]:
df_monthly_by_org1['attrition_rate'] = df_monthly_by_org1['n_terminated_employees'] / df_monthly_by_org1['avg_active_employees']

##### Monthly by org_l02

Deriving counts for active, terminated, and new hire employees

In [21]:
temp_indices = ['ds', 'ds_start', 'org_l00', 'org_l01', 'org_l02']

# deriving active employee count
df_monthly_by_org2 = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees') \

df_monthly_by_org2 = pd.merge(left=df_monthly_by_org2, right=new_hires_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org2['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_org2.n_new_hire_employees, errors='coerce')
df_monthly_by_org2['n_new_hire_employees'] = df_monthly_by_org2.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_org2 = pd.merge(left=df_monthly_by_org2, right=terminated_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org2['n_terminated_employees'] = pd.to_numeric(df_monthly_by_org2['n_terminated_employees'], errors='coerce')
df_monthly_by_org2['n_terminated_employees'] = df_monthly_by_org2['n_terminated_employees'].fillna(0).astype('int')


Deriving two-month average of active employees

In [22]:
df_temp_reindexed = df_monthly_by_org2 \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0,1,2], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_org2 = pd.merge(left=df_monthly_by_org2, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_org2['avg_active_employees'] = np.where(
    df_monthly_by_org2['avg_active_employees'].isnull(),
    df_monthly_by_org2['n_active_employees'],
    df_monthly_by_org2['avg_active_employees']
)

Deriving `attrition_rate`

In [23]:
df_monthly_by_org2['attrition_rate'] = df_monthly_by_org2['n_terminated_employees'] / df_monthly_by_org2['avg_active_employees']

#### Monthly results by level

Deriving the following by level
- `n_active_employees`, `n_new_hire_employees`, `n_terminated_employees`, `avg_active_employees`, `job_level_index`, `job_level_label`

In [24]:
temp_indices = ['ds', 'ds_start', 'job_level_category_ordered_w_indicators']

# deriving active employee count
df_monthly_by_level = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_level = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_level = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

df_monthly_by_level = pd.merge(left=df_monthly_by_level, right=new_hires_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_monthly_by_level['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_level.n_new_hire_employees, errors='coerce')
df_monthly_by_level['n_new_hire_employees'] = df_monthly_by_level.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_level = pd.merge(left=df_monthly_by_level, right=terminated_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_monthly_by_level['n_terminated_employees'] = pd.to_numeric(df_monthly_by_level['n_terminated_employees'], errors='coerce')
df_monthly_by_level['n_terminated_employees'] = df_monthly_by_level['n_terminated_employees'].fillna(0).astype('int')

df_temp_reindexed = df_monthly_by_level \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_level = pd.merge(left=df_monthly_by_level, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_level['avg_active_employees'] = np.where(
    df_monthly_by_level['avg_active_employees'].isnull(),
    df_monthly_by_level['n_active_employees'],
    df_monthly_by_level['avg_active_employees']
)

df_monthly_by_level[['job_level_index', 'job_level_label']] = df_monthly_by_level.job_level_category_ordered_w_indicators.str.split('--', n=1, expand=True)

df_monthly_by_level['attrition_rate'] = df_monthly_by_level['n_terminated_employees'] / df_monthly_by_level['avg_active_employees']


#### Monthly results by job track

Deriving the following by job track
- `n_active_employees`, `n_new_hire_employees`, `n_terminated_employees`, `avg_active_employees`, `job_level_index`, `job_level_label`

In [25]:
df_employees_plus.columns

Index(['ds_start', 'ds', 'is_last_day_of_year', 'employee_id', 'hire_date',
       'prehire_status', 'is_active', 'termination_date',
       'termination_date_coalesced', 'termination_month', 'termination_reason',
       'is_termination_voluntary', 'is_terminated', 'org_l00', 'org_l01',
       'org_l02', 'org_l03', 'gender', 'gender_remapped', 'ethnicity',
       'ethnicity_remapped', 'is_manager_track', 'job_track', 'job_level_idx',
       'job_level_category', 'job_level_category_ordered_w_indicators',
       'is_new_hire'],
      dtype='object')

In [26]:
df_employees_plus[['is_manager_track', 'job_track']].drop_duplicates()

,is_manager_track,job_track
0,True,M
72,False,S
144,False,IC


In [27]:
temp_indices = ['ds', 'ds_start', 'is_manager_track', 'job_track']

# deriving active employee count
df_monthly_by_track = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_level = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_level = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

df_monthly_by_track = pd.merge(left=df_monthly_by_track, right=new_hires_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_monthly_by_track['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_track.n_new_hire_employees, errors='coerce')
df_monthly_by_track['n_new_hire_employees'] = df_monthly_by_track.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_track = pd.merge(left=df_monthly_by_track, right=terminated_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_monthly_by_track['n_terminated_employees'] = pd.to_numeric(df_monthly_by_track['n_terminated_employees'], errors='coerce')
df_monthly_by_track['n_terminated_employees'] = df_monthly_by_track['n_terminated_employees'].fillna(0).astype('int')

df_temp_reindexed = df_monthly_by_track \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0,1], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_track = pd.merge(left=df_monthly_by_track, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_track['avg_active_employees'] = np.where(
    df_monthly_by_track['avg_active_employees'].isnull(),
    df_monthly_by_track['n_active_employees'],
    df_monthly_by_track['avg_active_employees']
)

df_monthly_by_track['track_category'] = df_monthly_by_track.is_manager_track.apply(lambda x: 'M' if x else 'IC/S')


## Annual results

### Annual (overall)

In [28]:
df_annual = pd.DataFrame({
    'ds': df_employees_plus_annual[df_employees_plus_annual.is_last_day_of_year].ds.unique(), 
}).sort_values('ds')

df_annual['ds_start'] = df_annual.ds.dt.to_period('Y').dt.start_time

# end of year n
active_employees_by_status = df_employees_plus_annual[df_employees_plus_annual['is_active']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_annual = df_annual.merge(right=active_employees_by_status, how='left', on='ds')

# start of year n
active_employees_by_status = df_employees_plus_annual[
    (df_employees_plus_annual.ds_start >= df_employees_plus_annual.hire_date)
    & (df_employees_plus_annual.ds <= df_employees_plus_annual.termination_date_coalesced)] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees_soy')
df_annual = df_annual.merge(right=active_employees_by_status, how='left', on='ds')

# n_terminated_employees
terminated_by_dates_annual_by_status = df_employees_plus_annual[df_employees_plus_annual.is_terminated] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

df_annual = df_annual.merge(right=terminated_by_dates_annual_by_status, how='left', on='ds')
df_annual['n_terminated_employees'] = pd.to_numeric(df_annual.n_terminated_employees, errors='coerce').fillna(0).astype('int')

# avg_active_employees
df_annual['avg_active_employees'] = (
    df_annual[['n_active_employees', 'n_active_employees_soy']].astype(float).mean(axis=1)
)

# attrition_rate
df_annual['attrition_rate'] = df_annual['n_terminated_employees'] / df_annual['avg_active_employees']

### Annual results by org

##### Annual by org_l01

In [29]:
temp_indices = ['ds', 'ds_start', 'org_l00', 'org_l01']
df_annual_by_org1 = df_employees_plus_annual[temp_indices].copy(True).drop_duplicates().sort_values(temp_indices).reset_index(drop=True)

df_annual_by_org1['ds_start'] = df_annual_by_org1.ds.dt.to_period('Y').dt.start_time

# end of year n ----------------------------------------------------------------------
active_employees_by_status = df_employees_plus_annual[df_employees_plus_annual['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_annual_by_org1 = df_annual_by_org1.merge(right=active_employees_by_status, how='left', on=temp_indices)

# adding placeholder field and reordering fields
df_annual_by_org1['org_l02'] = df_annual_by_org1['org_l01'] + ' Total'
df_annual_by_org1 = df_annual_by_org1[temp_indices + ['org_l02'] + ['n_active_employees']]

# start of year n --------------------------------------------------------------------
active_employees_by_status = df_employees_plus_annual[
    (df_employees_plus_annual.ds_start >= df_employees_plus_annual.hire_date)
    & (df_employees_plus_annual.ds <= df_employees_plus_annual.termination_date_coalesced)] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees_soy')
df_annual_by_org1 = df_annual_by_org1.merge(right=active_employees_by_status, how='left', on=temp_indices)

# n_terminated_employees ------------------------------------------------------
terminated_by_dates_annual_by_status = df_employees_plus_annual[df_employees_plus_annual.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

df_annual_by_org1 = df_annual_by_org1.merge(right=terminated_by_dates_annual_by_status, how='left', on=temp_indices)
df_annual_by_org1['n_terminated_employees'] = pd.to_numeric(df_annual_by_org1.n_terminated_employees, errors='coerce').fillna(0).astype('int')

# avg_active_employees ---------------------------------------------------------------
df_annual_by_org1['avg_active_employees'] = (
    df_annual_by_org1[['n_active_employees', 'n_active_employees_soy']].astype(float).mean(axis=1)
)

# attrition_rate ---------------------------------------------------------------------
df_annual_by_org1['attrition_rate'] = df_annual_by_org1['n_terminated_employees'] / df_annual_by_org1['avg_active_employees']


##### Annual by org_l02

In [30]:
temp_indices = ['ds', 'ds_start', 'org_l00', 'org_l01', 'org_l02']
df_annual_by_org2 = df_employees_plus_annual[temp_indices].copy(True).drop_duplicates().sort_values(temp_indices).reset_index(drop=True)

df_annual_by_org2['ds_start'] = df_annual_by_org2.ds.dt.to_period('Y').dt.start_time

# end of year n ----------------------------------------------------------------------
active_employees_by_status = df_employees_plus_annual[df_employees_plus_annual['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_annual_by_org2 = df_annual_by_org2.merge(right=active_employees_by_status, how='left', on=temp_indices)

# start of year n --------------------------------------------------------------------
active_employees_by_status = df_employees_plus_annual[
    (df_employees_plus_annual.ds_start >= df_employees_plus_annual.hire_date)
    & (df_employees_plus_annual.ds <= df_employees_plus_annual.termination_date_coalesced)] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees_soy')
df_annual_by_org2 = df_annual_by_org2.merge(right=active_employees_by_status, how='left', on=temp_indices)

# n_terminated_employees ------------------------------------------------------
terminated_by_dates_annual_by_status = df_employees_plus_annual[df_employees_plus_annual.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

df_annual_by_org2 = df_annual_by_org2.merge(right=terminated_by_dates_annual_by_status, how='left', on=temp_indices)
df_annual_by_org2['n_terminated_employees'] = pd.to_numeric(df_annual_by_org2.n_terminated_employees, errors='coerce').fillna(0).astype('int')

# avg_active_employees ---------------------------------------------------------------
df_annual_by_org2['avg_active_employees'] = (
    df_annual_by_org2[['n_active_employees', 'n_active_employees_soy']].astype(float).mean(axis=1)
)

# attrition_rate ---------------------------------------------------------------------
df_annual_by_org2['attrition_rate'] = df_annual_by_org2['n_terminated_employees'] / df_annual_by_org2['avg_active_employees']


##### Annual results by level

Deriving the following by level
- `n_active_employees`, `n_new_hire_employees`, `n_terminated_employees`, `avg_active_employees`, `job_level_index`, `job_level_label`

In [31]:
temp_indices = ['ds', 'ds_start', 'job_level_category_ordered_w_indicators']

# deriving active employee count
df_annual_by_level = df_employees_plus_annual[df_employees_plus_annual['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving new hire employee count
# new_hires_by_dates_monthly_by_status_and_level = df_employees_plus_annual[df_employees_plus_annual['is_new_hire']] \
#     .groupby(temp_indices)['employee_id'].nunique() \
#     .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_level = df_employees_plus_annual[df_employees_plus_annual.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_terminated_employees')

# df_annual_by_level = pd.merge(left=df_annual_by_level, right=new_hires_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
# df_annual_by_level['n_new_hire_employees'] = pd.to_numeric(df_annual_by_level.n_new_hire_employees, errors='coerce')
# df_annual_by_level['n_new_hire_employees'] = df_annual_by_level.n_new_hire_employees.fillna(0).astype('int')

df_annual_by_level = pd.merge(left=df_annual_by_level, right=terminated_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_annual_by_level['n_terminated_employees'] = pd.to_numeric(df_annual_by_level['n_terminated_employees'], errors='coerce')
df_annual_by_level['n_terminated_employees'] = df_annual_by_level['n_terminated_employees'].fillna(0).astype('int')

df_temp_reindexed = df_annual_by_level \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0], drop=True) \
    .reset_index(name='avg_active_employees')

df_annual_by_level = pd.merge(left=df_annual_by_level, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_annual_by_level['avg_active_employees'] = np.where(
    df_annual_by_level['avg_active_employees'].isnull(),
    df_annual_by_level['n_active_employees'],
    df_annual_by_level['avg_active_employees']
)

df_annual_by_level[['job_level_index', 'job_level_label']] = df_annual_by_level.job_level_category_ordered_w_indicators.str.split('--', n=1, expand=True)

df_annual_by_level['attrition_rate'] = df_annual_by_level['n_terminated_employees'] / df_annual_by_level['avg_active_employees']


## Attrition rate significance analysis

<summary>Monthly attrition significance POC (proof of concept)</summary>
<details>
    <code>
        df_2021_01_31 = df_monthly_by_org2[
            (df_monthly_by_org2.ds == '2021-01-31')
        ].sort_values('org_l02') \
            [['ds', 'org_l01', 'org_l02', 'n_active_employees', 'n_terminated_employees', 'attrition_rate']] \
            .reset_index(drop=True)
        df_2021_01_31['n_not_terminated_employees'] = df_2021_01_31['n_active_employees'] - df_2021_01_31['n_terminated_employees']

        total_terminated = df_2021_01_31['n_terminated_employees'].sum()
        total_not_terminated = df_2021_01_31['n_not_terminated_employees'].sum()

        results = []

        for index, row in df_2021_01_31.iterrows():
            # for org under review
            org_name = row['org_l02']
            terminated_org = row['n_terminated_employees']
            not_terminated_org = row['n_not_terminated_employees']
            
            # for other orgs
            terminated_rest = total_terminated - terminated_org
            not_terminated_rest = total_not_terminated - not_terminated_org
            
            # 2x2 contingency table
            # (org vs others) x (terminated vs not terminated)
            table = [[terminated_org, not_terminated_org],
                    [terminated_rest, not_terminated_rest]]
            
            odds_ratio, p_value = fisher_exact(table)
            
            results.append({
                'org_l02': org_name,
                'contingency_table': str(table),
                'odds_ratio': odds_ratio,
                'p_value': p_value
            })
            
        results_df = pd.DataFrame(results)
        results_df.sort_values(by='p_value')
    </code>
</details>

In [32]:
df_monthly_by_org_l02 = df_monthly_by_org2.copy(True)

df_monthly_by_org_l02['n_not_terminated_employees'] = df_monthly_by_org_l02['n_active_employees'] - df_monthly_by_org_l02['n_terminated_employees']

df_monthly_by_org_l02['contingency_table'] = ''
df_monthly_by_org_l02['odds_ratio'] = np.NaN
df_monthly_by_org_l02['p_value'] = np.NaN

for ds in df_monthly_by_org_l02.ds.unique():
    df_current = df_monthly_by_org_l02[df_monthly_by_org_l02.ds == ds].copy(True)
    total_terminated = df_current['n_terminated_employees'].sum()
    total_not_terminated = df_current['n_not_terminated_employees'].sum()

    for index, row in df_current.iterrows():
        # for org under review
        ds = row['ds']
        org_name = row['org_l02']
        terminated_org = row['n_terminated_employees']
        not_terminated_org = row['n_not_terminated_employees']
        
        # for other orgs
        terminated_rest = total_terminated - terminated_org
        not_terminated_rest = total_not_terminated - not_terminated_org
        
        # 2x2 contingency table
        # (org vs others) x (terminated vs not terminated)
        table = [[terminated_org, not_terminated_org],
                [terminated_rest, not_terminated_rest]]
        
        odds_ratio, p_value = fisher_exact(table)
        
        contingency_table = str(table)
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'contingency_table'
        ] = str(table)
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'odds_ratio'
        ] = odds_ratio
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'p_value'
        ] = p_value
df_monthly_by_org_l02['is_significant'] = df_monthly_by_org_l02['p_value'] <= 0.05

In [33]:
df_monthly_by_org_l02

,ds,ds_start,org_l00,org_l01,org_l02,n_active_employees,n_new_hire_employees,n_terminated_employees,avg_active_employees,attrition_rate,n_not_terminated_employees,contingency_table,odds_ratio,p_value,is_significant
0,2021-01-31,2021-01-01,Company Inc.,,,1,0,0,1.0,0.000000,1,"[[0, 1], [51, 1308]]",0.000000,1.000000,False
1,2021-01-31,2021-01-01,Company Inc.,Administrative,Communications,67,5,3,67.0,0.044776,64,"[[3, 64], [48, 1245]]",1.215820,0.736228,False
2,2021-01-31,2021-01-01,Company Inc.,Administrative,Finance,59,1,0,59.0,0.000000,59,"[[0, 59], [51, 1250]]",0.000000,0.165487,False
3,2021-01-31,2021-01-01,Company Inc.,Administrative,IT Services,74,6,1,74.0,0.013514,73,"[[1, 73], [50, 1236]]",0.338630,0.520868,False
4,2021-01-31,2021-01-01,Company Inc.,Administrative,Legal,57,6,2,57.0,0.035088,55,"[[2, 55], [49, 1254]]",0.930612,1.000000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1219,2026-12-31,2026-12-01,Company Inc.,Sales,Business,155,0,0,155.0,0.000000,155,"[[0, 155], [0, 1859]]",NaN,1.000000,False
1220,2026-12-31,2026-12-01,Company Inc.,Sales,Consumer,154,0,0,154.0,0.000000,154,"[[0, 154], [0, 1860]]",NaN,1.000000,False
1221,2026-12-31,2026-12-01,Company Inc.,Sales,Retail,156,0,0,156.0,0.000000,156,"[[0, 156], [0, 1858]]",NaN,1.000000,False
1222,2026-12-31,2026-12-01,Company Inc.,Sales,Sales Operations,177,0,0,177.0,0.000000,177,"[[0, 177], [0, 1837]]",NaN,1.000000,False


In [34]:
ds = '2022-02-28'
df_monthly_by_org_l02 \
    [df_monthly_by_org_l02.ds == ds] \
    [['ds', 'org_l00', 'org_l01', 'org_l02', 'attrition_rate', 'p_value', 'is_significant']] \
    .sort_values(['org_l00', 'org_l01', 'org_l02']) \
    .reset_index(drop=True) \
    .style.format({
        'ds': lambda t: t.strftime('%Y-%m-%d') if pd.notnull(t) else '',
        'attrition_rate': '{:.1%}',
        'p_value': '{:.2f}'
    })

,ds,org_l00,org_l01,org_l02,attrition_rate,p_value,is_significant
0,2022-02-28,Company Inc.,,,0.0%,1.00,False
1,2022-02-28,Company Inc.,Administrative,Communications,8.9%,0.23,False
2,2022-02-28,Company Inc.,Administrative,Finance,4.6%,0.80,False
3,2022-02-28,Company Inc.,Administrative,IT Services,12.3%,0.03,True
4,2022-02-28,Company Inc.,Administrative,Legal,8.2%,0.59,False
5,2022-02-28,Company Inc.,Administrative,People,2.6%,0.23,False
6,2022-02-28,Company Inc.,Administrative,Risk Management,5.3%,1.00,False
7,2022-02-28,Company Inc.,Production,Hardware,2.1%,0.08,False
8,2022-02-28,Company Inc.,Production,Quality Control,8.0%,0.49,False
9,2022-02-28,Company Inc.,Production,Research,2.5%,0.23,False


## Report build

### Headcount over time

In [35]:
report_n_months = 24

mask_ds = df_monthly_by_org2.ds.isin(df_monthly_by_org2[df_monthly_by_org2.ds_start <= current_month] \
    .ds.unique()[-report_n_months:])

df_monthly_by_org2[mask_ds][['ds', 'n_active_employees', 'n_new_hire_employees', 'n_terminated_employees']] \
    .groupby(by=['ds']).sum() \
    .reset_index(drop=False) \
    .style.format({
        'ds': lambda t: t.strftime('%Y-%m-%d') if pd.notnull(t) else '',
        'n_active_employees': '{:,}'
    })


,ds,n_active_employees,n_new_hire_employees,n_terminated_employees
0,2022-12-31,"1,824",105,73
1,2023-01-31,"1,883",85,29
2,2023-02-28,"1,921",119,84
3,2023-03-31,"1,823",8,103
4,2023-04-30,"1,726",9,103
5,2023-05-31,"1,682",63,108
6,2023-06-30,"1,663",83,106
7,2023-07-31,"1,686",95,72
8,2023-08-31,"1,774",93,0
9,2023-09-30,"1,850",138,62


### Current month headcount by org

In [36]:
# defining orgs that show org_l02 breakdown and order of iteration
org_l01 = {'Sales': True, 'Production': True, 'Administrative': False}

# filtering data to current month, with corresponding org indices of interest
current_month_headcount_by_org = df_monthly_by_org2[df_monthly_by_org2.ds_start == current_month][['org_l01', 'org_l02', 'n_active_employees']].groupby(by=['org_l01', 'org_l02']).sum().sort_values(by=['org_l01', 'org_l02']).reset_index()

current_month_headcount_by_org['percent_of_total'] = current_month_headcount_by_org['n_active_employees'] / current_month_headcount_by_org['n_active_employees'].sum()

# creating blank dataframe
display_headcount_by_org = pd.DataFrame([{'org_l01': '', 'org_l02': '', 'n_active_employees': 0, 'percent_of_total': 0}])

# iterate through orgs determined in the dictionary above, appending subtotals where necessary
for org in org_l01:
    if org_l01[org]:
        display_headcount_by_org = pd.concat([display_headcount_by_org, current_month_headcount_by_org[current_month_headcount_by_org.org_l01 == org]], ignore_index=True)
    org_n_active_employees = current_month_headcount_by_org[current_month_headcount_by_org.org_l01 == org].n_active_employees.sum()
    total_active_employees = df_employees_plus[(df_employees_plus.ds_start == current_month) & (df_employees_plus.is_active)].employee_id.nunique()
    percent_of_total = org_n_active_employees / total_active_employees
    subtotal_row = {'org_l01': f'{org} Total', 'org_l02': f'{org} Total', 'n_active_employees': org_n_active_employees, 'percent_of_total': org_n_active_employees / total_active_employees}
    display_headcount_by_org = pd.concat([display_headcount_by_org, pd.DataFrame([subtotal_row])], ignore_index=True)

total_row = {'org_l01': 'Company Total', 'org_l02': 'Company Total', 'n_active_employees': total_active_employees, 'percent_of_total': 1}

# dropping record of blank dataframe
display_headcount_by_org.drop(index=display_headcount_by_org[(display_headcount_by_org.org_l01 == '') & (display_headcount_by_org.org_l02 == '')].index, axis=0, inplace=True)

# adding total record
display_headcount_by_org = pd.concat([display_headcount_by_org, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine_active(row):
    n_active_formatted = f"{row['n_active_employees']:,}"
    if row['org_l02'] == 'Company Total':
        return f"{n_active_formatted}"
    perc_formatted = f"{row['percent_of_total']:.1%}"
    
    return f"{n_active_formatted} ({perc_formatted})"

# applying formatting to a single column
display_headcount_by_org['headcount_and_percent_of_total'] = display_headcount_by_org.apply(format_and_combine_active, axis=1)

# drop and rename columns
display_headcount_by_org.rename(columns={'org_l02': 'org'}, inplace=True)
display_headcount_by_org.drop(columns=['org_l01', 'n_active_employees', 'percent_of_total'], inplace=True)

display_headcount_by_org


,org,headcount_and_percent_of_total
0,Business,155 (7.7%)
1,Consumer,154 (7.6%)
2,Retail,156 (7.7%)
3,Sales Operations,177 (8.8%)
4,Sales Training,158 (7.8%)
5,Sales Total,800 (39.7%)
6,Hardware,133 (6.6%)
7,Quality Control,107 (5.3%)
8,Research,109 (5.4%)
9,Service Delivery,122 (6.1%)


### Headcount year-over-year

In [37]:
df_headcount_monthly_yoy = df_monthly.copy(True)
df_headcount_monthly_yoy['yyyy'] = df_headcount_monthly_yoy['ds'].dt.year
df_headcount_monthly_yoy['mm'] = df_headcount_monthly_yoy['ds'].dt.month
df_headcount_monthly_yoy['mmm'] = df_headcount_monthly_yoy['ds'].dt.strftime('%b')

mask_ds = (df_headcount_monthly_yoy.ds_start <= current_month) \
    & (df_headcount_monthly_yoy.ds_start >= '2022-01-01')

df_pivoted = df_headcount_monthly_yoy[mask_ds] \
    [['yyyy', 'mm', 'mmm', 'ds', 'ds_start', 'n_active_employees', 'avg_active_employees', 'attrition_rate']] \
    .pivot_table(index='yyyy', columns='mm', values='n_active_employees')

# create a mapping from month number to abbreviation then rename the columns
month_map = {i: pd.Timestamp(f'2023-{i:02d}-01').strftime('%b') for i in df_pivoted.columns}
df_pivoted = df_pivoted.rename(columns=month_map)

# create formatting for each column then display
# not necessary when displaying into plot
def num_or_blank(x):
    return '' if pd.isnull(x) else f'{int(x):,}'

formatters = {
    col: num_or_blank if str(col) else None
    for col in df_pivoted.columns
}

df_pivoted.style.format(formatters)


mm,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
yyyy,,,,,,,,,,,,
2022,"1,411","1,517","1,581","1,599","1,604","1,602","1,559","1,564","1,748","1,716","1,796","1,824"
2023,"1,883","1,921","1,823","1,726","1,682","1,663","1,686","1,774","1,850","1,891","1,966","1,943"
2024,"1,994","1,907","1,878","1,802","1,838","1,825","1,870","1,930","2,048","2,034","2,014",


### Current headcount by level

In [38]:
temp_indices = ['job_level_category_ordered_w_indicators', 'job_level_index', 'job_level_label']
# filtering data to current month, with corresponding org indices of interest
current_month_headcount_by_level = df_monthly_by_level[df_monthly_by_level.ds_start == current_month] \
    [temp_indices + ['n_active_employees']] \
    .groupby(by=temp_indices).sum() \
    .sort_values(by=temp_indices).reset_index()

current_month_headcount_by_level['percent_of_total'] = current_month_headcount_by_level['n_active_employees'] / current_month_headcount_by_level['n_active_employees'].sum()

total_active_employees = df_employees_plus[(df_employees_plus.ds_start == current_month) & (df_employees_plus.is_active)].employee_id.nunique()
total_row = {
    'job_level_category_ordered_w_indicators': 'Company Total', 
    'job_level_index': 'Company Total', 
    'job_level_label': 'Company Total', 
    'n_active_employees': total_active_employees, 
    'percent_of_total': 1
}

current_month_headcount_by_level = pd.concat([current_month_headcount_by_level, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine(row):
    n_active_formatted = f"{row['n_active_employees']:,}"
    if row['job_level_label'] == 'Company Total':
        return f"{n_active_formatted}"
    perc_formatted = f"{row['percent_of_total']:.1%}"
    
    return f"{n_active_formatted} ({perc_formatted})"

# applying formatting to a single column
current_month_headcount_by_level['headcount_and_percent_of_total'] = current_month_headcount_by_level.apply(format_and_combine, axis=1)

current_month_headcount_by_level.rename(columns={
    'job_level_label': 'Level',
    'headcount_and_percent_of_total': 'Headcount'
})[['Level', 'Headcount']]


,Level,Headcount
0,Support (S1-3),527 (26.2%)
1,Associate (IC1-3),578 (28.7%)
2,Senior (IC4-5),285 (14.2%)
3,Staff (IC 6-7),267 (13.3%)
4,Principal (IC 8-10),50 (2.5%)
5,Manager (M4-6),270 (13.4%)
6,Senior Manager (M7),27 (1.3%)
7,Director (M8-9),9 (0.4%)
8,SVP (M11),1 (0.0%)
9,Company Total,"2,014"


### Current headcount by job track

In [39]:
temp_indices = ['track_category']
# filtering data to current month, with corresponding org indices of interest
current_month_headcount_by_track = df_monthly_by_track[df_monthly_by_track.ds_start == current_month] \
    [temp_indices + ['n_active_employees']] \
    .groupby(by=temp_indices).sum() \
    .sort_values(by=temp_indices, ascending=False).reset_index()

current_month_headcount_by_track['percent_of_total'] = current_month_headcount_by_track['n_active_employees'] / current_month_headcount_by_track['n_active_employees'].sum()

total_active_employees = df_employees_plus[(df_employees_plus.ds_start == current_month) & (df_employees_plus.is_active)].employee_id.nunique()
total_row = {
    'track_category': 'Company Total', 
    'n_active_employees': total_active_employees, 
    'percent_of_total': 1
}

current_month_headcount_by_track = pd.concat([current_month_headcount_by_track, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine(row):
    n_active_formatted = f"{row['n_active_employees']:,}"
    if row['track_category'] == 'Company Total':
        return f"{n_active_formatted}"
    perc_formatted = f"{row['percent_of_total']:.1%}"
    
    return f"{n_active_formatted} ({perc_formatted})"

# applying formatting to a single column
current_month_headcount_by_track['headcount_and_percent_of_total'] = current_month_headcount_by_track.apply(format_and_combine, axis=1)

current_month_headcount_by_track.rename(columns={
    'track_category': 'Manager/IC',
    'headcount_and_percent_of_total': 'Headcount'
})[['Manager/IC', 'Headcount']]


,Manager/IC,Headcount
0,M,307 (15.2%)
1,IC/S,"1,707 (84.8%)"
2,Company Total,"2,014"


### Headcount year-over-year

In [40]:
df_headcount_monthly_yoy = df_monthly.copy(True)
df_headcount_monthly_yoy['yyyy'] = df_headcount_monthly_yoy['ds'].dt.year
df_headcount_monthly_yoy['mm'] = df_headcount_monthly_yoy['ds'].dt.month
df_headcount_monthly_yoy['mmm'] = df_headcount_monthly_yoy['ds'].dt.strftime('%b')

mask_ds = (df_headcount_monthly_yoy.ds_start <= current_month) \
    & (df_headcount_monthly_yoy.ds_start >= '2021-01-01')

df_headcount_pivoted = df_headcount_monthly_yoy[mask_ds] \
    [['yyyy', 'mm', 'mmm', 'ds', 'ds_start', 'n_active_employees', 'avg_active_employees', 'attrition_rate']] \
    .pivot_table(index='yyyy', columns='mm', values='n_active_employees')

# create a mapping from month number to abbreviation then rename the columns
month_map = {i: pd.Timestamp(f'2023-{i:02d}-01').strftime('%b') for i in df_headcount_pivoted.columns}
df_headcount_pivoted = df_headcount_pivoted.rename(columns=month_map)

# create formatting for each column then display
# not necessary when displaying into plot
def int_or_blank(x):
    return '' if pd.isnull(x) else f'{int(x):,}'

formatters = {
    col: int_or_blank if str(col) else None
    for col in df_headcount_pivoted.columns
}

df_headcount_pivoted.style.format(formatters)


mm,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
yyyy,,,,,,,,,,,,
2021,"1,360","1,414","1,353","1,450","1,484","1,542","1,491","1,502","1,498","1,483","1,465","1,457"
2022,"1,411","1,517","1,581","1,599","1,604","1,602","1,559","1,564","1,748","1,716","1,796","1,824"
2023,"1,883","1,921","1,823","1,726","1,682","1,663","1,686","1,774","1,850","1,891","1,966","1,943"
2024,"1,994","1,907","1,878","1,802","1,838","1,825","1,870","1,930","2,048","2,034","2,014",


### Attrition year-over-year

In [41]:
df_attrition_monthly_yoy = df_monthly.copy(True)
df_attrition_monthly_yoy['yyyy'] = df_attrition_monthly_yoy['ds'].dt.year
df_attrition_monthly_yoy['mm'] = df_attrition_monthly_yoy['ds'].dt.month
df_attrition_monthly_yoy['mmm'] = df_attrition_monthly_yoy['ds'].dt.strftime('%b')

mask_ds = (df_attrition_monthly_yoy.ds_start <= current_month) \
    & (df_attrition_monthly_yoy.ds_start >= '2022-01-01')

df_attrition_pivoted = df_attrition_monthly_yoy[mask_ds] \
    [['yyyy', 'mm', 'mmm', 'ds', 'ds_start', 'n_active_employees', 'avg_active_employees', 'attrition_rate']] \
    .pivot_table(index='yyyy', columns='mm', values='attrition_rate')

# create a mapping from month number to abbreviation then rename the columns
month_map = {i: pd.Timestamp(f'2023-{i:02d}-01').strftime('%b') for i in df_attrition_pivoted.columns}
df_attrition_pivoted = df_attrition_pivoted.rename(columns=month_map)

# create formatting for each column then display
# not necessary when displaying into plot
def percent_or_blank(x):
    return '' if pd.isnull(x) else f'{x:.1%}'

formatters = {
    col: percent_or_blank if str(col) else None
    for col in df_attrition_pivoted.columns
}

df_attrition_pivoted.style.format(formatters)


mm,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
yyyy,,,,,,,,,,,,
2022,3.6%,6.2%,0.0%,6.9%,2.7%,2.9%,3.0%,1.0%,0.0%,2.4%,6.4%,4.0%
2023,1.6%,4.4%,5.5%,5.8%,6.3%,6.3%,4.3%,0.0%,3.4%,3.7%,2.7%,2.5%
2024,2.9%,5.4%,5.5%,6.6%,4.7%,4.3%,0.0%,0.0%,0.8%,4.3%,1.5%,


### Attrition by org

#### Calculate current and previous month results

In [42]:
# defining orgs that show org_l02 breakdown and order of iteration
org_map = [
    {'org_name': 'Sales', 'incl_org_l02': True},
    {'org_name': 'Production', 'incl_org_l02': True},
    {'org_name': 'Administrative', 'incl_org_l02': False},
]

selected_fields = ['org_l01','org_l02','n_active_employees','n_terminated_employees','avg_active_employees','attrition_rate']

default_values = {'org_l01': '',
                  'org_l02': '',
                  'n_active_employees': 0,
                  'n_terminated_employees': 0,
                  'avg_active_employees': 0,
                  'attrition_rate': 0}

attrition_by_org_current_month = pd.DataFrame([default_values])
attrition_by_org_previous_month = pd.DataFrame([default_values])

def prep_monthly_table(df, ds):
    for org_idx in org_map:
        # calculate values related to org_l02
        if org_idx['incl_org_l02']:
            df = pd.concat([
                df,
                df_monthly_by_org2[(df_monthly_by_org2.ds_start == ds) & (df_monthly_by_org2.org_l01 == org_idx['org_name'])][selected_fields],
            ], ignore_index=True)
        # calculate subtotals that correspond to org_l01
        df = pd.concat([
                df,
                df_monthly_by_org1[(df_monthly_by_org1.ds_start == ds) & (df_monthly_by_org1.org_l01 == org_idx['org_name'])][selected_fields]],
            ignore_index=True
        )
    # calculate grand total
    df = pd.concat([
        df,
        pd.merge(
            left=pd.DataFrame([{'ds': pd.to_datetime(df_monthly[df_monthly.ds_start == ds].ds.iloc[0]), 'ds_start': pd.to_datetime(df_monthly[df_monthly.ds_start == ds].ds_start.iloc[0]), 'org_l01': 'Company Total', 'org_l02': 'Company Total'}]),
            right=df_monthly[df_monthly.ds_start == ds],
            how='left',
            on=['ds', 'ds_start']
        )[selected_fields]
    ], ignore_index=True)
    return df

attrition_by_org_current_month = prep_monthly_table(attrition_by_org_current_month, current_month)
attrition_by_org_previous_month = prep_monthly_table(attrition_by_org_previous_month, previous_month)


#### Calculate current year results

In [43]:
selected_fields = ['org_l01', 'org_l02', 'n_active_employees', 'n_terminated_employees', 'avg_active_employees', 'attrition_rate']

attrition_by_org_current_year = pd.DataFrame([default_values])

for org_idx in org_map:
    # calculate values related to org_l02
    if org_idx['incl_org_l02']:
        attrition_by_org_current_year = pd.concat([
            attrition_by_org_current_year,
            df_annual_by_org2[(df_annual_by_org2.ds_start == start_of_year) & (df_annual_by_org2.org_l01 == org_idx['org_name'])][selected_fields],
        ], ignore_index=True)
    # calculate subtotals that correspond to org_l01
    attrition_by_org_current_year = pd.concat([
        attrition_by_org_current_year,
        df_annual_by_org1[(df_annual_by_org1.ds_start == start_of_year) & (df_annual_by_org1.org_l01 == org_idx['org_name'])][selected_fields],
    ], ignore_index=True)
# calculate grand total
attrition_by_org_current_year = pd.concat([
    attrition_by_org_current_year,
    pd.merge(
        left=pd.DataFrame([{'ds': pd.to_datetime(df_annual[df_annual.ds_start == start_of_year].ds.iloc[0]), 'ds_start': pd.to_datetime(df_annual[df_annual.ds_start == start_of_year].ds_start.iloc[0]), 'org_l01': 'Company Total', 'org_l02': 'Company Total'}]),
        right=df_annual[df_annual.ds_start == start_of_year],
        how='left',
        on=['ds', 'ds_start']
    )[selected_fields]
    ], ignore_index=True)


#### Format attrition by org table

In [44]:
selected_fields = ['org_l01', 'org_l02', 'attrition_rate', 'n_terminated_employees']

attrition_by_org_table = attrition_by_org_current_month[selected_fields].copy(True)

attrition_by_org_table = pd.merge(
    left=attrition_by_org_table,
    right=attrition_by_org_previous_month[selected_fields],
    how='left',
    on=['org_l01', 'org_l02'],
    suffixes=['_curr', '_prev']
)

attrition_by_org_table = pd.merge(
    left=attrition_by_org_table,
    right=attrition_by_org_current_year[selected_fields],
    how='left',
    on=['org_l01', 'org_l02']
)

def format_and_combine_attrition_month(row):
    n_terminated_formatted = f"{row['n_terminated_employees_curr']:,}"
    perc_formatted = f"{row['attrition_rate_curr']:.1%}"
    return f"{perc_formatted} ({n_terminated_formatted})"

def format_and_combine_attrition_year(row):
    n_terminated_formatted = f"{row['n_terminated_employees']:,}"
    perc_formatted = f"{row['attrition_rate']:.1%}"
    return f"{perc_formatted} ({n_terminated_formatted})"

attrition_by_org_table[current_month_formatted] = attrition_by_org_table.apply(format_and_combine_attrition_month, axis=1)
attrition_by_org_table['month_delta'] = attrition_by_org_table['attrition_rate_curr'] - attrition_by_org_table['attrition_rate_prev']
attrition_by_org_table['M/M ∆'] = (attrition_by_org_table['month_delta'].apply(lambda x: f'{x:.1%}'))

attrition_by_org_table[attrition_year_label] = attrition_by_org_table.apply(format_and_combine_attrition_year, axis=1)

attrition_by_org_table['Org'] = attrition_by_org_table.org_l02

drop_row = attrition_by_org_table[(attrition_by_org_table.org_l01 == '') & (attrition_by_org_table.org_l02 == '')]
attrition_by_org_table.drop(index=drop_row.index, inplace=True)


In [45]:
attrition_by_org_table = pd.merge(
    left=attrition_by_org_table[['Org', current_month_formatted, 'M/M ∆', attrition_year_label]],
    right=df_monthly_by_org_l02[['org_l02', 'p_value', 'is_significant']][df_monthly_by_org_l02.ds_start == current_month],
    how='left',
    left_on='Org',
    right_on='org_l02'
)

attrition_by_org_table.drop(columns={'org_l02'})

,Org,Nov '24,M/M ∆,2024 + KFA,p_value,is_significant
0,Business,1.3% (2),-3.1%,45.0% (58),1.000000,False
1,Consumer,1.9% (3),-3.8%,41.6% (53),0.727609,False
2,Retail,1.9% (3),0.0%,45.2% (57),0.729115,False
3,Sales Operations,3.4% (6),-2.1%,32.9% (50),0.048793,True
4,Sales Training,1.3% (2),-2.5%,44.5% (59),1.000000,False
5,Sales Total,2.0% (16),-2.3%,41.5% (277),NaN,NaN
6,Hardware,0.7% (1),-1.5%,32.7% (37),0.718243,False
7,Quality Control,0.0% (0),-5.5%,55.1% (51),0.407064,False
8,Research,1.8% (2),-4.4%,40.0% (37),0.683123,False
9,Service Delivery,0.0% (0),-5.6%,31.7% (32),0.255908,False


### Attrition detail

#### Exit reasons

In [143]:
temp_indices = ['termination_month', 'termination_reason']
mask = (df_employees_plus['is_terminated'])

df_monthly_by_termination_reason = df_employees_plus[mask].groupby(temp_indices)['employee_id'].nunique().reset_index(name='n_terminated_monthly')

# temp_indices = ['termination_year']
temp_indices = ['termination_year', 'termination_reason']
mask = (df_employees_plus_annual['is_terminated'])
df_annual_by_termination_reason = df_employees_plus_annual[mask].groupby(temp_indices)['employee_id'].nunique().reset_index(name='n_terminated_annually')

termination_reasons_table = df_annual_by_termination_reason[df_annual_by_termination_reason.termination_year == start_of_year].iloc[:,1:]

termination_reasons_table = pd.merge(
    left=termination_reasons_table,
    right=df_monthly_by_termination_reason.copy(True) \
        [df_monthly_by_termination_reason.termination_month == current_month].iloc[:,1:]
    )

sort_indices = ['n_terminated_annually', 'n_terminated_monthly']
sort_indices.sort(reverse=True)
termination_reasons_table.sort_values(by=sort_indices, ascending=False, inplace=True)

total_row = pd.DataFrame([{
    'termination_reason': 'Respondents',
    'n_terminated_annually': df_annual_by_termination_reason[df_annual_by_termination_reason.termination_year == start_of_year].n_terminated_annually.sum(),
    'n_terminated_monthly': df_monthly_by_termination_reason[df_monthly_by_termination_reason.termination_month == current_month].n_terminated_monthly.sum(),
}])

termination_reasons_table = pd.concat([termination_reasons_table,total_row], ignore_index=True)

termination_reasons_table['percent_of_annual'] = termination_reasons_table.n_terminated_annually / total_row.n_terminated_annually.sum()
termination_reasons_table['percent_of_monthly'] = termination_reasons_table.n_terminated_monthly / total_row.n_terminated_monthly.sum()

def format_and_combine(row):
    n_size = f"{row.iloc[2]:,}"
    perc = f"{row.iloc[1]:.0%}"
    return n_size if row.iloc[0] == 'Respondents' else f'{perc} ({n_size})'

termination_reasons_table['exits_and_percent_annual'] = termination_reasons_table[['termination_reason', 'percent_of_annual', 'n_terminated_annually']].apply(format_and_combine, axis=1)
termination_reasons_table['exits_and_percent_monthly'] = termination_reasons_table[['termination_reason', 'percent_of_monthly', 'n_terminated_monthly']].apply(format_and_combine, axis=1)

termination_reasons_table = termination_reasons_table[['termination_reason', 'exits_and_percent_monthly', 'exits_and_percent_annual']].rename(
    columns={'termination_reason': 'Top Exit Reasons',
             'exits_and_percent_monthly': current_month_formatted,
             'exits_and_percent_annual': attrition_year_label_2})

pd.concat([
    termination_reasons_table.head(5),
    termination_reasons_table.tail(1)
], ignore_index=True)


,Top Exit Reasons,Nov '24,2024 YTD
0,Career Change,13% (4),4% (30)
1,Limited Internal Mobility Options,10% (3),5% (32)
2,Company Confidence,10% (3),4% (29)
3,Gross Misconduct,6% (2),6% (38)
4,Conflict with Work Location / Return to Work,6% (2),5% (37)
5,Respondents,31,687


#### Current attrition by level

In [48]:
temp_indices = ['job_level_category_ordered_w_indicators', 'job_level_index', 'job_level_label']
# filtering data to current month, with corresponding org indices of interest
current_month_attrition_by_level = df_monthly_by_level[df_monthly_by_level.ds_start == current_month] \
    [temp_indices + ['attrition_rate', 'n_terminated_employees']] \
    .groupby(by=temp_indices).sum() \
    .sort_values(by=temp_indices).reset_index()

total_attrition_rate = df_monthly[df_monthly.ds_start == current_month].attrition_rate.iloc[0]
total_terminated = df_monthly[df_monthly.ds_start == current_month].n_terminated_employees.iloc[0]
total_row = {
    'job_level_category_ordered_w_indicators': 'Company Total', 
    'job_level_index': 'Company Total', 
    'job_level_label': 'Company Total', 
    'attrition_rate': total_attrition_rate, 
    'n_terminated_employees': total_terminated
}

current_month_attrition_by_level = pd.concat([current_month_attrition_by_level, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine(row):
    attrition_rate_formatted = f"{row['attrition_rate']:.1%}"
    n_terminated_formatted = f"{row['n_terminated_employees']:,}"
    return f"{attrition_rate_formatted} ({n_terminated_formatted})"

# applying formatting to a single column
current_month_attrition_by_level['rate_and_n'] = current_month_attrition_by_level.apply(format_and_combine, axis=1)

attrition_by_level_table = current_month_attrition_by_level.rename(columns={
    'job_level_label': 'Level',
    'rate_and_n': current_month_formatted
})[['Level', current_month_formatted]]


In [49]:
temp_indices = ['job_level_category_ordered_w_indicators', 'job_level_index', 'job_level_label']
# filtering data to current month, with corresponding org indices of interest
current_year_attrition_by_level = df_annual_by_level[df_annual_by_level.ds_start == start_of_year] \
    [temp_indices + ['attrition_rate', 'n_terminated_employees']] \
    .groupby(by=temp_indices).sum() \
    .sort_values(by=temp_indices).reset_index()

total_attrition_rate = df_annual[df_annual.ds_start == start_of_year].attrition_rate.iloc[0]
total_terminated = df_annual[df_annual.ds_start == start_of_year].n_terminated_employees.iloc[0]
total_row = {
    'job_level_category_ordered_w_indicators': 'Company Total', 
    'job_level_index': 'Company Total', 
    'job_level_label': 'Company Total', 
    'attrition_rate': total_attrition_rate, 
    'n_terminated_employees': total_terminated
}

current_year_attrition_by_level = pd.concat([current_year_attrition_by_level, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine(row):
    attrition_rate_formatted = f"{row['attrition_rate']:.1%}"
    n_terminated_formatted = f"{row['n_terminated_employees']:,}"
    return f"{attrition_rate_formatted} ({n_terminated_formatted})"

# applying formatting to a single column
current_year_attrition_by_level['rate_and_n'] = current_year_attrition_by_level.apply(format_and_combine, axis=1)

attrition_by_level_table = pd.merge(
    left=attrition_by_level_table,
    right=current_year_attrition_by_level.rename(columns={
        'job_level_label': 'Level',
        'rate_and_n': attrition_year_label
    })[['Level', attrition_year_label]]
)

attrition_by_level_table

,Level,Nov '24,2024 + KFA
0,Support (S1-3),1.9% (10),33.7% (173)
1,Associate (IC1-3),1.7% (10),35.6% (202)
2,Senior (IC4-5),1.0% (3),38.0% (108)
3,Staff (IC 6-7),1.1% (3),32.6% (85)
4,Principal (IC 8-10),0.0% (0),44.0% (22)
5,Manager (M4-6),1.8% (5),32.2% (85)
6,Senior Manager (M7),0.0% (0),23.5% (6)
7,Director (M8-9),0.0% (0),54.5% (6)
8,SVP (M11),0.0% (0),0.0% (0)
9,Company Total,1.5% (31),40.8% (687)
